# Install dependencies from uv and setup reloading of imports.

In [17]:
!uv sync
%load_ext autoreload
%autoreload 2

Resolved 133 packages in 5ms
Checked 130 packages in 23ms
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Create the agents

In [18]:
from google.adk.agents import Agent
from google.adk.tools import google_search, agent_tool

import config
import tools
import callbacks
import instructions

weather_agent = Agent(
    name="weather_agent",
    model=config.GEMINI_MODEL,
    instruction=instructions.get_agent_instructions("weather-agent-instructions"),
    before_model_callback=callbacks.log_agent_name_before_callback,
    tools=[tools.get_weather, tools.get_lat_lon]
)

search_agent = Agent(name="search_agent",
                     model=config.GEMINI_MODEL,
                     instruction=instructions.get_agent_instructions("search-agent-instructions"),
                     before_model_callback=callbacks.log_agent_name_before_callback,
                     tools=[google_search])

root_agent = Agent(
    name="root_agent",
    model=config.GEMINI_MODEL,
    mode="chat",
    instruction=instructions.get_agent_instructions("challenge-3-router-agent-instructions"),
    before_model_callback=callbacks.log_agent_name_before_callback,
    tools=[agent_tool.AgentTool(agent=search_agent)],
    sub_agents=[weather_agent]
)

# Setup the agent tester

In [19]:
import agent_tester

tester = agent_tester.AgentTester(root_agent)

In [20]:
print("================ New York ===========================")
await tester.run_prompt("What is the weather for New York City, New York")
print("================ Reston =============================")
await tester.run_prompt("What is the weather for Reston, VA")
print("================ Los Angeles ========================")
await tester.run_prompt("What is the weather for Los Angeles, CA")

================ New York ===========================
Calling agent root_agent
Calling agent weather_agent
Calling agent weather_agent
Calling agent weather_agent
📍 Weather for New York City, New York 📅 Friday, July 26, 2024                 

🌤️ Conditions:     Slight Chance Showers And Thunderstorms 🌡️ Temperature:      
86°F  |  High: 86°  Low: 77° 💧 Humidity:       N/A 💨 Wind:           9 mph    
South, then 5 to 8 mph Southwest 🌧️ Precipitation:  20% chance of showers and   
thunderstorms (afternoon), 80% chance of showers and thunderstorms (tonight) 👁️ 
Visibility:     N/A 🌅 Sunrise:        N/A 🌇 Sunset:         N/A               
================ Reston =============================
Calling agent root_agent
Calling agent weather_agent
Calling agent weather_agent
Calling agent weather_agent
📍 Weather for Reston, VA 📅 Friday, July 26, 2024                              

🌤️ Conditions: Scattered Showers And Thunderstorms 🌡️ Temperature: 90°F | High: 
90°F Low: 71°F 💨 Wind: 8 mph SW 

# Perform tests for searching

In [21]:
print("================ Time ===========================")
await tester.run_prompt("What is the current time in Tokyo")
print("================ News ===========================")
await tester.run_prompt("Tell me the top news story from yesterday.")

================ Time ===========================
Calling agent root_agent
Calling agent search_agent
Calling agent root_agent
The current time in Tokyo, Japan, is 5:20 AM on Saturday, August 8, 2026. Tokyo 
is in the Japan Time (JST) time zone, which is UTC+09:00, and does not observe  
daylight saving time.                                                           
================ News ===========================
Calling agent root_agent
Calling agent search_agent
Calling agent root_agent
On Thursday, August 6, 2026, several significant news stories unfolded.         

One prominent development in US politics involved the American Israel Public    
Affairs Committee (AIPAC) reportedly considering a substantial financial        
campaign to support the Republican Senate nominee in Michigan. This comes after 
their unsuccessful $32 million effort to boost Representative Haley Stevens     
against Dr. Abdul El-Sayed in the Democratic primary, a result that drew a new  
attack from the 